### E4

#### Mision 1

In [1]:
import pandas as pd
import plotly.express as px

import dash
from dash import html, dcc, Input, Output, State

In [2]:
rides = pd.read_csv("E4_datos/cta_ridership_2022_2024.csv")
stations = pd.read_csv("E4_datos/cta_stations.csv")

# limpieza minima
rides["date"] = pd.to_datetime(rides["date"])
rides["rides"] = pd.to_numeric(rides["rides"], errors="coerce")

# unir los datos de rides con stations para obtener info. adicional
df = rides.merge(stations, on="station_id", how="left")

# nombre
df["station_label"] = df["station_descriptive_name"].fillna(df["stationname"])

df.head()

,station_id,stationname,date,daytype,rides,daytype_name,station_name,station_descriptive_name,lines,ada,...,longitude,red,blue,g,brn,p,y,pnk,o,station_label
0,40010,Austin-Forest Park,2022-01-01,A,231,Saturday,Austin,Austin (Blue Line),Blue,False,...,-87.776812,False,True,False,False,False,False,False,False,Austin (Blue Line)
1,40020,Harlem-Lake,2022-01-01,A,500,Saturday,Harlem/Lake,Harlem/Lake (Green Line),Green,True,...,-87.803176,False,False,True,False,False,False,False,False,Harlem/Lake (Green Line)
2,40030,Pulaski-Lake,2022-01-01,A,284,Saturday,Pulaski,Pulaski (Green Line),Green,True,...,-87.725404,False,False,True,False,False,False,False,False,Pulaski (Green Line)
3,40040,Quincy/Wells,2022-01-01,A,493,Saturday,Quincy/Wells,"Quincy/Wells (Brown, Orange, Purple & Pink lines)","Brown, Purple, Pink, Orange",True,...,-87.633740,False,False,False,True,True,False,True,True,"Quincy/Wells (Brown, Orange, Purple & Pink lines)"
4,40050,Davis,2022-01-01,A,502,Saturday,Davis,Davis (Purple Line),Purple,True,...,-87.683543,False,False,False,False,True,False,False,False,Davis (Purple Line)


In [3]:
df.isna().sum()

station_id                  0
stationname                 0
date                        0
daytype                     0
rides                       0
daytype_name                0
station_name                0
station_descriptive_name    0
lines                       0
ada                         0
latitude                    0
longitude                   0
red                         0
blue                        0
g                           0
brn                         0
p                           0
y                           0
pnk                         0
o                           0
station_label               0
dtype: int64

In [4]:
min_date = df["date"].min()
max_date = df["date"].max()

In [5]:
# opciones del selector de estaciones
station_options = (
    df[["station_id", "station_label"]].drop_duplicates().sort_values("station_label")
)

station_options = []

for _, row in df[["station_id", "station_label"]].iterrows():
    station_options.append({"label": row["station_label"], "value": row["station_id"]})

In [6]:
station_options[:5]

[{'label': 'Austin (Blue Line)', 'value': 40010},
 {'label': 'Harlem/Lake (Green Line)', 'value': 40020},
 {'label': 'Pulaski (Green Line)', 'value': 40030},
 {'label': 'Quincy/Wells (Brown, Orange, Purple & Pink lines)',
  'value': 40040},
 {'label': 'Davis (Purple Line)', 'value': 40050}]

Vamos con el Dash.

In [7]:
app = dash.Dash(__name__)

app.layout = html.Div(
    style={"fontFamily": "Arial", "margin": "30px"},
    children=[
        html.H1("CTA L: demanda por estación"),

        html.Label("selecciona una estación:"),
        dcc.Dropdown(
            id="station-dropdown",
            options=station_options,
            value=station_options[0]["value"],
            clearable=True
        ),

        html.Br(),

        html.Label("selecciona rango de fechas: (Rango sugerido de fechas: {} a {})".format(min_date.date(), max_date.date())),
        dcc.DatePickerRange(
            id="date-range",
            start_date=min_date.date(),
            end_date=max_date.date(),
            display_format="YYYY-MM-DD"
        ),

        html.Br(),
        html.Br(),

        html.Button("Actualizar", id="update-button", n_clicks=0),

        html.Hr(),

        html.Div(id="summary-card"),

        dcc.Graph(id="rides-graph")
    ]
)

Hay varios comandos y métodos útiles que usamos y se recomienda investigar. 

Ahora vamos con el callback, que es para interactuar.

In [8]:
from datetime import date

df["date_only"] = df["date"].dt.date

In [9]:
@app.callback(
    Output("summary-card", "children"), # actualiza
    Output("rides-graph", "figure"),
    Input("update-button", "n_clicks"), # escucha cambios, es el disparador
    State("station-dropdown", "value"), # permiten leer valores de componentes sin disparar el callback
    State("date-range", "start_date"),
    State("date-range", "end_date")
)

def update_dashboard(n_clicks, station_id, start_date, end_date):

    if station_id is None:
        fig = px.line(title="selecciona una estación")
        return html.P("no se ha seleccionado estación."), fig

    if start_date is None or end_date is None:
        fig = px.line(title="selecciona un rango de fechas")
        return html.P("falta seleccionar fecha inicial o final."), fig

    try:
        start_date = date.fromisoformat(start_date[:10])
        end_date = date.fromisoformat(end_date[:10])
    except ValueError:
        fig = px.line(title="fecha inválida")
        return html.P("la fecha seleccionada no es válida."), fig

    station_name = df[df["station_id"] == station_id]["station_label"].iloc[0]

    filtros = html.Div([
        html.P("filtros activos:"),
        html.P(f"estación: {station_name}"),
        html.P(f"período: {start_date} a {end_date}")
    ])

    if start_date > end_date:
        fig = px.line(title="Rango de fechas inválido")
        return html.Div([
            filtros,
            html.P("la fecha inicial debe ser menor o igual a la fecha final.")
        ]), fig

    dff = df[
        (df["station_id"] == station_id) &
        (df["date_only"] >= start_date) &
        (df["date_only"] <= end_date)
    ].copy()

    if dff.empty:
        fig = px.line(title=f"Sin datos para {station_name}")

        summary = html.Div(
            style={
                "border": "1px solid #ddd",
                "padding": "15px",
                "borderRadius": "8px",
                "marginBottom": "20px"
            },
            children=[
                filtros,
                html.H3("Sin datos"),
                html.P("los filtros seleccionados no retornan registros.")
            ]
        )

        return summary, fig

    dff = dff.sort_values("date")

    total_rides = int(dff["rides"].sum())
    avg_rides = dff["rides"].mean()

    max_row = dff.loc[dff["rides"].idxmax()]
    max_day = max_row["date"].strftime("%Y-%m-%d")
    max_rides = int(max_row["rides"])

    predominant_daytype = dff["daytype_name"].mode().iloc[0]

    summary = html.Div(
        style={
            "border": "1px solid #ddd",
            "padding": "15px",
            "borderRadius": "8px",
            "marginBottom": "20px"
        },
        children=[
            filtros,
            html.H3(station_name),
            html.P(f"total de entradas: {total_rides:,}"),
            html.P(f"promedio diario de entradas: {avg_rides:.1f}"),
            html.P(f"día con mayor cantidad de entradas: {max_day} ({max_rides:,})"),
            html.P(f"tipo de día predominante: {predominant_daytype}")
        ]
    )

    fig = px.line(
        dff,
        x="date",
        y="rides",
        title=f"Entradas diarias en {station_name} ({start_date} a {end_date})",
        labels={
            "date": "Fecha",
            "rides": "Entradas"
        }
    )

    return summary, fig

In [10]:
# app.run(port=8050, debug=False)

#### Mision 2

In [11]:
app = dash.Dash(__name__)

app.layout = html.Div(
    style={"fontFamily": "Arial", "margin": "30px"},
    children=[
        html.H1("CTA L: comparación entre estaciones"),

        html.Label("selecciona dos o más estaciones:"),
        dcc.Dropdown(
            id="stations-dropdown",
            options=station_options,
            value=[station_options[0]["value"], station_options[1]["value"]],
            multi=True,
            placeholder="Selecciona estaciones"
        ),

        html.Br(),

        html.Label("selecciona rango de fechas:"),
        dcc.DatePickerRange(
            id="date-range-m2",
            start_date=min_date.date(),
            end_date=max_date.date(),
            display_format="YYYY-MM-DD"
        ),

        html.Br(),
        html.Br(),

        html.Button("actualizar comparación", id="update-button-m2", n_clicks=0),

        html.Hr(),

        html.H2("evolución diaria"),
        dcc.Graph(id="comparison-line-graph"),

        html.H2("total de entradas por estación"),
        dcc.Graph(id="comparison-bar-graph"),

        html.H2("resumen por estación"),
        html.Div(id="comparison-summary")
    ]
)

In [12]:
def calcular_variacion_primera_ultima_semana(dff_station):
    # esto calcula la variacion porcentual entre el promedio de la primera semana
    # y el promedio de la última semana del período seleccionado.

    dff_station = dff_station.sort_values("date")

    fecha_inicio = dff_station["date"].min()
    fecha_fin = dff_station["date"].max()

    primera_semana_fin = fecha_inicio + pd.Timedelta(days=6)
    ultima_semana_inicio = fecha_fin - pd.Timedelta(days=6)

    primera_semana = dff_station[
        (dff_station["date"] >= fecha_inicio) &
        (dff_station["date"] <= primera_semana_fin)
    ]

    ultima_semana = dff_station[
        (dff_station["date"] >= ultima_semana_inicio) &
        (dff_station["date"] <= fecha_fin)
    ]

    promedio_primera = primera_semana["rides"].mean()
    promedio_ultima = ultima_semana["rides"].mean()

    if pd.isna(promedio_primera) or promedio_primera == 0:
        return None

    variacion = ((promedio_ultima - promedio_primera) / promedio_primera) * 100

    return variacion

In [13]:
@app.callback(
    Output("comparison-line-graph", "figure"),
    Output("comparison-bar-graph", "figure"),
    Output("comparison-summary", "children"),
    Input("update-button-m2", "n_clicks"),
    State("stations-dropdown", "value"),
    State("date-range-m2", "start_date"),
    State("date-range-m2", "end_date")
)

def update_comparison(n_clicks, station_ids, start_date, end_date):

    if station_ids is None or len(station_ids) < 2:
        fig_line = px.line(title="selecciona dos o más estaciones")
        fig_bar = px.bar(title="selecciona dos o más estaciones")
        message = html.P("debes seleccionar al menos dos estaciones para comparar.")
        return fig_line, fig_bar, message

    if start_date is None or end_date is None:
        fig_line = px.line(title="selecciona un rango de fechas")
        fig_bar = px.bar(title="selecciona un rango de fechas")
        message = html.P("falta seleccionar fecha inicial o final.")
        return fig_line, fig_bar, message

    try:
        start_date = date.fromisoformat(start_date[:10])
        end_date = date.fromisoformat(end_date[:10])
    except ValueError:
        fig_line = px.line(title="Fecha inválida")
        fig_bar = px.bar(title="Fecha inválida")
        message = html.P("La fecha seleccionada no es válida.")
        return fig_line, fig_bar, message

    if start_date > end_date:
        fig_line = px.line(title="Rango de fechas inválido")
        fig_bar = px.bar(title="Rango de fechas inválido")
        message = html.P("La fecha inicial debe ser menor o igual a la fecha final.")
        return fig_line, fig_bar, message

    dff = df[
        (df["station_id"].isin(station_ids)) &
        (df["date_only"] >= start_date) &
        (df["date_only"] <= end_date)
    ].copy()

    filtros = html.Div([
        html.P("filtros activos:"),
        html.P(f"cantidad de estaciones seleccionadas: {len(station_ids)}"),
        html.P(f"período: {start_date} a {end_date}")
    ])

    if dff.empty:
        fig_line = px.line(title="Sin datos para los filtros seleccionados")
        fig_bar = px.bar(title="Sin datos para los filtros seleccionados")

        message = html.Div(
            style={
                "border": "1px solid #ddd",
                "padding": "15px",
                "borderRadius": "8px",
                "marginBottom": "15px"
            },
            children=[
                filtros,
                html.H3("Sin datos"),
                html.P("Los filtros seleccionados no retornan registros.")
            ]
        )

        return fig_line, fig_bar, message

    dff = dff.sort_values("date")

    # gráfico 1: evolución diaria
    daily = (
        dff.groupby(["date", "station_label"], as_index=False)["rides"].sum()
    )

    fig_line = px.line(
        daily,
        x="date",
        y="rides",
        color="station_label",
        title=f"Evolución diaria de entradas por estación ({start_date} a {end_date})",
        labels={
            "date": "Fecha",
            "rides": "Entradas diarias",
            "station_label": "Estación"
        }
    )

    # gráfico 2: total por estación
    totals = (
        dff.groupby("station_label", as_index=False)["rides"].sum().sort_values("rides", ascending=False)
    )

    fig_bar = px.bar(
        totals,
        x="station_label",
        y="rides",
        title=f"Total de entradas por estación ({start_date} a {end_date})",
        labels={
            "station_label": "Estación",
            "rides": "Total de entradas"
        }
    )

    # resumen por estación
    cards = [filtros]

    for station_id in station_ids:
        dff_station = dff[dff["station_id"] == station_id].copy()

        if dff_station.empty:
            continue

        station_name = dff_station["station_label"].iloc[0]

        total_rides = int(dff_station["rides"].sum())
        avg_daily = dff_station["rides"].mean()
        max_daily = int(dff_station["rides"].max())

        variation = calcular_variacion_primera_ultima_semana(dff_station)

        if variation is None:
            variation_text = "No calculable"
        else:
            variation_text = f"{variation:.1f}%"

        card = html.Div(
            style={
                "border": "1px solid #ddd",
                "padding": "15px",
                "borderRadius": "8px",
                "marginBottom": "15px"
            },
            children=[
                html.H3(station_name),
                html.P(f"total de entradas: {total_rides:,}"),
                html.P(f"promedio diario: {avg_daily:.1f}"),
                html.P(f"máximo diario: {max_daily:,}"),
                html.P(f"variación primera vs. última semana: {variation_text}")
            ]
        )

        cards.append(card)

    return fig_line, fig_bar, cards

In [14]:
# app.run(port=8051, debug=False)

### M3

In [ ]:
line_columns = {
    "Red": "red",
    "Blue": "blue",
    "Green": "g",
    "Brown": "brn",
    "Purple": "p",
    "Yellow": "y",
    "Pink": "pnk",
    "Orange": "o"
}

line_options = [{"label": "Todas", "value": "all"}] + [
    {"label": line, "value": line}
    for line in line_columns.keys()
]

def is_true_column(series):
    return series.astype(str).str.lower() == "true"

def filter_by_line(data, selected_line):
    if selected_line == "all":
        return data.copy()

    column = line_columns[selected_line]
    return data[is_true_column(data[column])].copy()

def make_station_options(selected_line):
    data = filter_by_line(df, selected_line)

    options_df = (
        data[["station_id", "station_label"]].drop_duplicates().sort_values("station_label")
    )

    return [
        {"label": row["station_label"], "value": row["station_id"]}
        for _, row in options_df.iterrows()
    ]


def format_ada(value):
    if pd.isna(value):
        return "Sin información"

    value = str(value).lower()

    if value == "true":
        return "Sí"
    elif value == "false":
        return "No"
    else:
        return value

In [16]:
app = dash.Dash(__name__)

app.layout = html.Div(
    style={"fontFamily": "Arial", "margin": "30px"},
    children=[
        html.H1("CTA L: exploración complementaria"),

        html.Label("filtrar por línea:"),
        dcc.Dropdown(
            id="line-dropdown-m3",
            options=line_options,
            value="all",
            clearable=False
        ),

        html.Br(),

        html.Label("selecciona una estación:"),
        dcc.Dropdown(
            id="station-dropdown-m3",
            options=make_station_options("all"),
            value=make_station_options("all")[0]["value"],
            clearable=False
        ),

        html.Br(),

        html.Label("Selecciona rango de fechas:"),
        dcc.DatePickerRange(
            id="date-range-m3",
            start_date=min_date,
            end_date=max_date,
            display_format="YYYY-MM-DD"
        ),

        html.Br(),
        html.Br(),

        html.Button("Actualizar", id="update-button-m3", n_clicks=0),

        html.Hr(),

        html.H2("Información de estación"),
        html.Div(id="station-info-m3"),

        html.H2("Ranking de estaciones"),
        dcc.Graph(id="ranking-graph-m3"),

        html.H2("Patrón por tipo de día"),
        dcc.Graph(id="daytype-graph-m3")
    ]
)

In [17]:
@app.callback(
    Output("station-dropdown-m3", "options"),
    Output("station-dropdown-m3", "value"),
    Input("line-dropdown-m3", "value")
)
def update_station_dropdown(selected_line):

    options = make_station_options(selected_line)

    if len(options) == 0:
        return [], None

    return options, options[0]["value"]

In [ ]:
@app.callback(
    Output("station-info-m3", "children"),
    Output("ranking-graph-m3", "figure"),
    Output("daytype-graph-m3", "figure"),
    Input("update-button-m3", "n_clicks"),
    State("line-dropdown-m3", "value"),
    State("station-dropdown-m3", "value"),
    State("date-range-m3", "start_date"),
    State("date-range-m3", "end_date")
)

def update_m3(n_clicks, selected_line, station_id, start_date, end_date):

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # datos filtrados por línea y fecha
    data_line = filter_by_line(df, selected_line)

    data_period = data_line[
        (data_line["date"] >= start_date) &
        (data_line["date"] <= end_date)
    ].copy()

    if data_period.empty:
        empty_bar = px.bar(title="Sin datos para los filtros seleccionados")
        empty_daytype = px.bar(title="Sin datos para los filtros seleccionados")
        return html.P("No hay datos para esta selección."), empty_bar, empty_daytype

    # información de estación

    station_data = data_period[data_period["station_id"] == station_id].copy()

    if station_id is None or station_data.empty:
        station_info = html.P("Selecciona una estación válida.")
    else:
        station_name = station_data["station_label"].iloc[0]
        lines = station_data["lines"].iloc[0]
        ada = format_ada(station_data["ada"].iloc[0])

        latitude = station_data["latitude"].iloc[0]
        longitude = station_data["longitude"].iloc[0]

        total_rides = int(station_data["rides"].sum())
        avg_rides = station_data["rides"].mean()

        station_info = html.Div(
            style={
                "border": "1px solid #ddd",
                "padding": "15px",
                "borderRadius": "8px",
                "marginBottom": "20px"
            },
            children=[
                html.H3(station_name),
                html.P(f"Líneas: {lines}"),
                html.P(f"Accesibilidad ADA: {ada}"),
                html.P(f"Latitud: {latitude}"),
                html.P(f"Longitud: {longitude}"),
                html.P(f"Total de entradas en el período: {total_rides:,}"),
                html.P(f"Promedio diario: {avg_rides:.1f}")
            ]
        )


    # ranking de estaciones

    ranking = (
        data_period
        .groupby("station_label", as_index=False)["rides"]
        .sum()
        .sort_values("rides", ascending=False)
        .head(10)
    )

    fig_ranking = px.bar(
        ranking,
        x="station_label",
        y="rides",
        title="Top 10 estaciones por entradas totales",
        labels={
            "station_label": "Estación",
            "rides": "Total de entradas"
        }
    )


    # patrón por tipo de día
    if station_id is not None and not station_data.empty:
        daytype_data = (
            station_data
            .groupby("daytype_name", as_index=False)["rides"]
            .mean()
        )

        title_daytype = "Promedio diario de entradas por tipo de día"
    else:
        daytype_data = (
            data_period
            .groupby("daytype_name", as_index=False)["rides"]
            .mean()
        )

        title_daytype = "Promedio diario de entradas por tipo de día"

    fig_daytype = px.bar(
        daytype_data,
        x="daytype_name",
        y="rides",
        title=title_daytype,
        labels={
            "daytype_name": "Tipo de día",
            "rides": "Promedio de entradas"
        }
    )

    return station_info, fig_ranking, fig_daytype

In [19]:
app.run(port=8052, debug=False)